In [0]:
from pyspark.sql import functions as F

state_map = {
    "alabama":"AL","alaska":"AK","arizona":"AZ","arkansas":"AR","california":"CA",
    "colorado":"CO","connecticut":"CT","delaware":"DE","florida":"FL","georgia":"GA",
    "hawaii":"HI","idaho":"ID","illinois":"IL","indiana":"IN","iowa":"IA",
    "kansas":"KS","kentucky":"KY","louisiana":"LA","maine":"ME","maryland":"MD",
    "massachusetts":"MA","michigan":"MI","minnesota":"MN","mississippi":"MS",
    "missouri":"MO","montana":"MT","nebraska":"NE","nevada":"NV",
    "new hampshire":"NH","new jersey":"NJ","new mexico":"NM","new york":"NY",
    "north carolina":"NC","north dakota":"ND","ohio":"OH","oklahoma":"OK",
    "oregon":"OR","pennsylvania":"PA","rhode island":"RI","south carolina":"SC",
    "south dakota":"SD","tennessee":"TN","texas":"TX","utah":"UT","vermont":"VT",
    "virginia":"VA","washington":"WA","west virginia":"WV","wisconsin":"WI",
    "wyoming":"WY","district of columbia":"DC"}

map_expr = F.create_map([F.lit(x) for kv in state_map.items() for x in kv])

def norm_state(col):
    """Full name -> code; 2-letter codes pass through; junk -> null.
    DEBT: belongs in Silver — move during wk5 pipeline consolidation."""
    lower = F.lower(F.trim(col))
    return F.when(F.length(F.trim(col)) == 2, F.upper(F.trim(col))) \
            .otherwise(map_expr[lower])

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sp = spark.table("jobmarket.silver.silver_job_postings")
sk = spark.table("jobmarket.silver.silver_posting_skills")


# --- base: role x state counts + disclosed-salary median ---
base = (sp
    .filter(F.col("title_norm").isNotNull() & (F.col("title_norm") != ""))
    .withColumn("state", F.coalesce(norm_state(F.col("state")), F.lit("Unknown")))
    .withColumn("salary_mid",
        F.when(F.col("salary_is_estimated") == False,
            F.when(F.col("salary_max").isNotNull(),
                   (F.col("salary_min") + F.col("salary_max")) / 2)
             .otherwise(F.col("salary_min")))))   # estimated -> null -> excluded from median

role_state = (base
    .groupBy("title_norm", "state")
    .agg(F.countDistinct("posting_id").alias("posting_count"),
         F.round(F.expr("percentile_approx(salary_mid, 0.5)")).alias("median_salary"),
         F.count("salary_mid").alias("salary_sample")))

# --- top 10 technical skills per role x state: window rank -> collect ---
role_skills = (sk
    .filter(F.col("skill_group") != "soft")
    .join(sp.select("posting_id", "title_norm",
                    F.coalesce(norm_state(F.col("state")), F.lit("Unknown")).alias("state")),
          "posting_id")
    .groupBy("title_norm", "state", "skill")
    .agg(F.countDistinct("posting_id").alias("n")))

w = Window.partitionBy("title_norm", "state").orderBy(F.desc("n"), "skill")

top_skills = (role_skills
    .withColumn("rk", F.row_number().over(w))
    .filter("rk <= 10")
    .groupBy("title_norm", "state")
    .agg(F.collect_list("skill").alias("top_skills")))   # list preserves rank order

# --- assemble; trend deliberately null until the time axis matures (wk5) ---
role_summary = (role_state
    .join(top_skills, ["title_norm", "state"], "left")
    .withColumn("demand_trend", F.lit(None).cast("string")))

(role_summary.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.gold.gold_role_summary"))

g = spark.table("jobmarket.gold.gold_role_summary")
print("Rows:", g.count())

# the app's actual future query, as the sanity check:
g.filter("title_norm = 'data engineer' AND posting_count >= 5") \
 .orderBy(F.desc("posting_count")).limit(10).display()

In [0]:
spark.table("jobmarket.gold.gold_role_summary") \
    .filter("title_norm = 'software engineer' AND state = 'Unknown'") \
    .select("state", "posting_count", "salary_sample", "top_skills") \
    .display()